# ClimX recovery and D01 daily-quantile study

Francisco Angulo de Lafuente · ORCID [0009-0001-1634-7063](https://orcid.org/0009-0001-1634-7063).

The original climate emulator has two completed official Kaggle entries. This archive recovers that history and preserves a new, preregistered daily maximum-temperature study with verified CPU arithmetic. The GPU feasibility pilot is queued as of 10 November 2026,12:23 UTC; its result and the full scientific comparison are pending.

| Evidence | Verified result | Scope |
|---|---|---|
| Existing Kaggle entry 57777154 | COMPLETE, public score 0.03070 | Existing 64-member baseline; not a D01 score |
| Existing Kaggle entry 57777260 | COMPLETE, public score 0.02852 | Existing 32-member control |
| D01 CPU verification | Six numerical fixtures passed; 5570 runtime files, 40 raw-band slices and 9 input files checked | CPU arithmetic and artifact identity |
| D01 GPU/scientific result | Pending | No new official entry or measured quality gain |

The old README's cancelled 64-member worker was followed by a successful worker completing all 48 bands. The local recovered parquet and the official entry records are separate evidence: Kaggle does not expose an uploaded-content hash here. See [OFFICIAL_HISTORY.json](OFFICIAL_HISTORY.json).

## One fixed daily-field hypothesis

D01 adds three quadratic greenhouse-gas features to the original conditional daily tasmax quantile model, with a fixed penalty of 100 training-year equivalents. The linear control uses the same data, seasonal transform and predictors. Training uses historical, SSP126 andSSP585; SSP370 is excluded from fitting and used for the prospective comparison on eight fixed full-resolution latitude bands. Earlier studies have already examined these scenarios, so this is not an untouched external confirmation.

```mermaid
flowchart LR
  A[Training daily tasmax] --> B[Fixed linear quantile control]
  A --> C[Fixed quadratic GHG quantile model]
  D[SSP370 daily validation] --> E[Paired daily loss comparison]
  B --> E
  C --> E
  E --> F[Separate daily-realisation and official-index study if screen passes]
```

The screening metric is truncated continuous ranked probability score in Kelvin, obtained from quantile pinball loss over probabilities 0.01 through0.99. The fixed gate requires at least 0.25% reduction and a positive lower 95% endpoint from a paired 10-year circular-block bootstrap (10000 replicates, seed20261110). This is a univariate distribution screen on selected bands, rather than the official regional 15-index metric. A screening pass requires a separate daily-realisation and official-index evaluation before candidate promotion. The [protocol](PROTOCOL.md) gives the full scope and limitations.

## Current rules and reproducibility

The [official rules](https://www.kaggle.com/competitions/climx-main-track/rules) require daily-field outputs followed by official index computation. Recovered neural index-residual families are excluded from this candidate path. Training here uses only organizer-provided data; no CMIP6-pretrained model is used. Public code is also shared through the associated Kaggle notebook, as required by section 3.6b.

The immutable source package retains the seven original emulator files unchanged and includes the new GPU sufficient-statistic model and runner. CPU and CUDA arithmetic must match the NumPy reference. The final CPU report matches the exact source and input manifests. Single-attempt execution markers are atomic; bytecode reads use an empty prefix. The immutable implementation keeps explicit paths for this audited experiment. [REPRODUCE.md](REPRODUCE.md) provides installation, data-store construction, data-free CPU checks and input/runtime-manifest regeneration commands. The helper creates a separate path-adapted copy with its own identity; the archived source stays unchanged. Its CPU fixtures and full existing-store fingerprint/verify flow were executed successfully, as recorded in [CPU_REPRODUCTION.json](CPU_REPRODUCTION.json). A fresh full-data download and clean dependency installation were not repeated.

## Attribution and archive contents

The distributional forecasting engine is adapted from [maverickss26's public Kaggle notebook](https://www.kaggle.com/code/maverickss26/climx-sota-main-track), credited in the retained Apache-2.0 [NOTICE](source/NOTICE) and[LICENSE](source/LICENSE). The new D01 fitting and screening code is by Francisco Angulo de Lafuente under Apache-2.0. The original separate official ClimX repository keeps its MIT licence; that tree is not bundled in this source package.

This archive contains source, protocol, hashes and aggregate verification. Competition observations, prediction maps, trained weights, credentials and private contact details are absent. The CPU-only Kaggle notebook materialises the identical source archive; it is a backup, not a competition submission.


# Reproduce the CPU checks and prepare another audited run

The original `source/` and its manifests are immutable execution artifacts. This guide creates a separate reproduction directory, changes only its three storage/cache/output paths, and assigns that copy a new source identity. The scientific algorithms and protocol stay unchanged. The helper runs CPU modes only; the Kaggle notebook provides the source ZIP rather than re-running the study.

## 1. Install a CPU environment and replay the numerical checks

Use Python3.11 or later in a new virtual environment. From the extracted archive root, on Linux:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements-cpu.txt
.venv/bin/python tools/reproduce_cpu.py fixtures --work /tmp/climx-d01-reproduction
```

On a Windows computer, run these Linux commands inside WSL2 or another Linux environment; the archived runner uses the Unix `resource` module. See the [official PyTorch installation guide](https://pytorch.org/get-started/locally/) to select the appropriate CPU or CUDA build for that Linux environment. `requirements-cpu.txt` gives the API floor, not a claim that every version combination was tested. The exact audited versions and original six-fixture result are recorded in `CPU_VERIFICATION.json`. The helper creates its own `cpu-fixtures/PREFLIGHT.json`, including its current runtime versions and new source-manifest SHA256. It preserves the archived source. No competition observations or GPU are needed for these fixtures.

The six checks cover quantile extraction, the linear ridge solution, the quadratic ridge solution, daily interpolation, pinball integration and training-only curvature standardisation. Passing them establishes software arithmetic, not a climate-quality gain or an official score.

## 2. Build the private full-resolution store

This step is for registered ClimX participants using the organizer-provided data under the [competition rules](https://www.kaggle.com/competitions/climx-main-track/rules). The full release is approximately200GB; processing requires additional disk space and time. Keep observations, caches and predictions private. This download is separate from the small data-free CPU replay above.

Install the two additional data-access packages in the same project environment:

```bash
.venv/bin/python -m pip install netCDF4 huggingface_hub
```

Create `build_private_store.py` outside the immutable source, using the original public data-access implementation:

```python
import sys
sys.path.insert(0, 'source')
from climx_sota.data import INDEX_VARS, repack_remote

if __name__ == '__main__':
    repack_remote(
        '/your/data-drive/climx/store_full',
        '/your/data-drive/climx/raw',
        band_rows=4, variables=INDEX_VARS,
        keep_raw=True, workers=4,
    )
```

Run it from the extracted archive root with the project Python. `repack_remote` uses the organizer's full-resolution Hugging Face release and original int16 codec. It retains raw files with `keep_raw=True`. The output needs the original192x288grid,48bands, all four training scenarios, forcings, annual global temperatures and calendars. This build path is provided from the retained engine; a fresh full200GBdownload was not repeated for this audit because the complete store was already present.

## 3. Regenerate input/runtime pins and verify the final copy

For the full study, use a CUDA-capable PyTorch environment with its installed NVIDIA library packages and select the version appropriate to your GPU using the official guide. The following command still uses CPU only. It fingerprints the private store's fixed40 band slices, input files and critical runtime files, writes new `INPUTS.json` into the reproduction copy, rebuilds its source manifest, then executes the final verification against that exact identity:

```bash
.venv/bin/python tools/reproduce_cpu.py preflight \
  --work /your/data-drive/climx-d01-reproduction \
  --store /your/data-drive/climx/store_full
```

The resulting `final-cpu-verify/PREFLIGHT.json` must say `passed:true` and match the new source/input manifest hashes. It does not need to match the original machine's runtime hashes. Changing data, runtime or source requires a new reviewed reproduction identity; retain the original archive and all negative or failed results.

## 4. GPU admission and scientific result

The frozen original GPU pilot is queued, with no GPU result in this archive. For another run, use the copied source directory with the normal resource-managed GPU queue, the new manifest SHA, and the protocol's fixed bounds. The runner modes are `pilot` and, only after a passing pilot, `screen`; the respective wall caps are10m and45m. Single-attempt markers live under the copied work directory's `results/` folder. The copied runner's interpreter must be the project environment containing CUDA PyTorch. Keep the prescribed parameters, validation split and gate unchanged.

A screening pass allows preparation of a separate daily-realisation and official-index comparison. Truncated CRPS on daily tasmax quantiles does not establish a leaderboard improvement or authorise a test submission.


In [ ]:
import base64, hashlib, io, json, pathlib, zipfile
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='412b28ea509efc62db7351c37bc957de9411b5d213d28f73a6af29d7da3acf98'
archive=pathlib.Path.cwd()/'climx-d01-source-archive-20261110.zip'
archive.write_bytes(payload)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    pins=json.loads(z.read('FILES_SHA256.json'))
    assert set(z.namelist())==set(pins)|{'FILES_SHA256.json'}
    assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in pins.items())
print(json.dumps({'artifact':archive.name,'bytes':len(payload),'sha256':hashlib.sha256(payload).hexdigest(),'kind':'source backup; no competition submission'}))
